In [ ]:
# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip install -q -U \
    langchain \
    langchain-core \
    langchain-google-genai \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu


# ============================================================
# 2. IMPORTS
# ============================================================

from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnableLambda

from langchain_core.tools import tool

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import TextLoader

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain.agents import create_agent


# ============================================================
# 3. GET GEMINI API KEY
# ============================================================

# In Colab:
# Secrets -> add:
#
# GOOGLE_API_KEY
#
# Value = your Gemini API key

api_key = userdata.get("GOOGLE_API_KEY")

if api_key is None:
    raise ValueError(
        "GOOGLE_API_KEY not found. "
        "Add it to Colab Secrets first."
    )

print("Gemini API key loaded successfully!")


# ============================================================
# 4. CREATE GEMINI CHAT MODEL
# ============================================================

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0
)

print("Gemini model loaded successfully!")


# ============================================================
# 5. BASIC MODEL CALL
# ============================================================

response = model.invoke(
    "Explain what LangChain is in one sentence."
)

print(response.content)


# ============================================================
# 6. MESSAGES
# ============================================================

messages = [
    SystemMessage(
        content="You are a helpful AI teacher."
    ),
    HumanMessage(
        content="Explain Transformers simply."
    )
]

response = model.invoke(messages)

print(response.content)


# ============================================================
# 7. RUNNABLE
# ============================================================

def uppercase(text):
    return text.upper()


uppercase_runnable = RunnableLambda(uppercase)

result = uppercase_runnable.invoke(
    "hello langchain"
)

print(result)


# ============================================================
# 8. RUNNABLE CHAIN
# ============================================================

def add_exclamation(text):
    return text + "!!!"


exclamation_runnable = RunnableLambda(
    add_exclamation
)

chain = (
    uppercase_runnable
    | exclamation_runnable
)

result = chain.invoke("hello langchain")

print(result)


# ============================================================
# 9. PROMPT TEMPLATE
# ============================================================

prompt = PromptTemplate.from_template(
    """
    Explain {topic} in simple terms
    with one small example.
    """
)

formatted_prompt = prompt.invoke(
    {"topic": "Self-Attention"}
)

print(formatted_prompt)


# ============================================================
# 10. PROMPT + MODEL
# ============================================================

chain = (
    prompt
    | model
)

response = chain.invoke(
    {"topic": "Embeddings"}
)

print(response.content)


# ============================================================
# 11. CHAT PROMPT TEMPLATE
# ============================================================

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert AI teacher."
    ),
    (
        "human",
        "Explain {topic} simply."
    )
])

chain = (
    chat_prompt
    | model
)

response = chain.invoke(
    {"topic": "RAG"}
)

print(response.content)


# ============================================================
# 12. OUTPUT PARSER
# ============================================================

parser = StrOutputParser()

chain = (
    chat_prompt
    | model
    | parser
)

result = chain.invoke(
    {"topic": "Vector Database"}
)

print(result)

print(type(result))


# ============================================================
# 13. COMPLETE LCEL CHAIN
# ============================================================

prompt = ChatPromptTemplate.from_template(
    """
    Explain the following topic for a beginner.

    Topic: {topic}

    Give:
    1. Definition
    2. How it works
    3. Small example
    """
)

chain = (
    prompt
    | model
    | StrOutputParser()
)

result = chain.invoke(
    {"topic": "Attention Mechanism"}
)

print(result)


# ============================================================
# 14. STREAMING
# ============================================================

for chunk in model.stream(
    "Explain Transformers in about 100 words."
):
    print(chunk.content, end="", flush=True)


# ============================================================
# 15. DOCUMENT LOADER
# ============================================================

# Create a small text file for demonstration

with open("sample.txt", "w") as f:
    f.write(
        """
        LangChain is a framework for building
        applications powered by language models.

        RAG allows an LLM to retrieve information
        from external documents.

        Embeddings convert text into numerical vectors.

        Vector databases store embeddings and allow
        similarity search.
        """
    )


loader = TextLoader("sample.txt")

documents = loader.load()

print(documents)

print("Number of documents:", len(documents))

print("Content:")
print(documents[0].page_content)

print("Metadata:")
print(documents[0].metadata)


# ============================================================
# 16. TEXT SPLITTER
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = splitter.split_documents(
    documents
)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i} ---")
    print(chunk.page_content)


# ============================================================
# 17. EMBEDDINGS
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

query_vector = embeddings.embed_query(
    "What is RAG?"
)

print("Vector length:", len(query_vector))

print("First 10 values:")
print(query_vector[:10])


# ============================================================
# 18. VECTOR STORE - FAISS
# ============================================================

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created!")


# ============================================================
# 19. SIMILARITY SEARCH
# ============================================================

results = vectorstore.similarity_search(
    "What is RAG?",
    k=2
)

for i, doc in enumerate(results):

    print(f"\n--- RESULT {i} ---")

    print(doc.page_content)


# ============================================================
# 20. RETRIEVER
# ============================================================

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 2
    }
)

retrieved_docs = retriever.invoke(
    "How does RAG work?"
)

for doc in retrieved_docs:

    print("\n--- RETRIEVED ---")
    print(doc.page_content)


# ============================================================
# 21. BASIC RAG
# ============================================================

def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


def retrieve_and_format(question):

    docs = retriever.invoke(question)

    return format_docs(docs)


rag_prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using ONLY the provided context.

    If the answer is not present in the context,
    say "I don't know based on the provided context."

    Context:
    {context}

    Question:
    {question}
    """
)


rag_chain = (
    {
        "context": retrieve_and_format,
        "question": lambda x: x
    }
    | rag_prompt
    | model
    | StrOutputParser()
)


answer = rag_chain.invoke(
    "What is RAG?"
)

print(answer)


# ============================================================
# 22. TOOLS
# ============================================================

@tool
def add(a: int, b: int) -> int:
    """
    Add two numbers.
    """
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers.
    """
    return a * b


tools = [
    add,
    multiply
]

print(add.invoke({
    "a": 10,
    "b": 20
}))

print(multiply.invoke({
    "a": 10,
    "b": 20
}))


# ============================================================
# 23. BIND TOOLS TO GEMINI
# ============================================================

model_with_tools = model.bind_tools(
    tools
)

response = model_with_tools.invoke(
    "What is 25 multiplied by 48?"
)

print(response)


# ============================================================
# 24. AGENT
# ============================================================

agent = create_agent(
    model=model,
    tools=tools
)

result = agent.invoke({
    "messages": [
        HumanMessage(
            content="What is 25 multiplied by 48?"
        )
    ]
})

print(
    result["messages"][-1].content
)


# ============================================================
# 25. SIMPLE CHAT MEMORY
# ============================================================

chat_history = []


# First message
user_message = HumanMessage(
    content="My name is Chidambaram."
)

chat_history.append(user_message)

response = model.invoke(
    chat_history
)

print("AI:", response.content)

chat_history.append(response)


# Second message
user_message = HumanMessage(
    content="What is my name?"
)

chat_history.append(user_message)

response = model.invoke(
    chat_history
)

print("AI:", response.content)


# ============================================================
# 26. CHAT MEMORY WITH PROMPT TEMPLATE
# ============================================================

conversation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful AI assistant."
    ),

    MessagesPlaceholder(
        variable_name="history"
    ),

    (
        "human",
        "{input}"
    )
])


conversation_chain = (
    conversation_prompt
    | model
    | StrOutputParser()
)


# First conversation
history = [
    HumanMessage(
        content="My favorite programming language is Python."
    ),
    AIMessage(
        content="That's great! Python is widely used in AI and machine learning."
    )
]


response = conversation_chain.invoke({
    "history": history,
    "input": "What is my favorite programming language?"
})

print(response)


# ============================================================
# 27. REAL CHAT LOOP
# ============================================================

chat_history = []

print("\nChat started!")
print("Type 'exit' to stop.\n")


while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    chat_history.append(
        HumanMessage(
            content=user_input
        )
    )

    response = model.invoke(
        chat_history
    )

    print(
        "AI:",
        response.content
    )

    chat_history.append(
        response
    )